# THINGS ResNet-50 Baseline on Colab

Clean Colab run for the image-only baseline. This notebook assumes the runtime is temporary and may delete `/content/human-things` when `RESET_LOCAL_REPO = True`.

Every long cell prints the paths, command, and expected next step so we can follow progress in Colab.

Default flow:
1. Check GPU.
2. Clone a fresh repo checkout.
3. Install requirements.
4. Restore THINGS data from Drive zip, or fall back to OSF.
5. Verify data completeness.
6. Build metadata and image splits.
7. Dry-run data loading.
8. Run a bounded smoke training pass.
9. Optionally run full training, extract embeddings, and evaluate them.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/sabszh/human-things.git"
PROJECT_ROOT = Path("/content/human-things")

RESET_LOCAL_REPO = True
USE_DRIVE_DATA_ZIP = True
DRIVE_DATA_ZIP = Path("/content/drive/MyDrive/human-things-data/human-things-data.zip")
DRIVE_DATA_FILE_ID = "1OofSEPS34SA6Jol3OIqO208ekIHz1UEF"
LOCAL_DATA_ZIP = Path("/content/human-things-data.zip")

DOWNLOAD_IMAGES = True
FORCE_FETCH = False  # Set True only if a local OSF file is corrupt; it redownloads requested files.

RUN_SHORT_TRAINING = True
RUN_FULL_TRAINING = False
RUN_EXTRACT_AND_EVAL = False
COPY_OUTPUTS_TO_DRIVE = True

DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/human-things-runs")

print("Configuration")
print("- repo:", REPO_URL)
print("- project root:", PROJECT_ROOT)
print("- reset local repo:", RESET_LOCAL_REPO)
print("- use Drive zip:", USE_DRIVE_DATA_ZIP)
print("- mounted Drive zip:", DRIVE_DATA_ZIP)
print("- shared Drive file id:", DRIVE_DATA_FILE_ID)
print("- local fallback zip:", LOCAL_DATA_ZIP)
print("- short training:", RUN_SHORT_TRAINING)
print("- full training:", RUN_FULL_TRAINING)
print("- extract/eval:", RUN_EXTRACT_AND_EVAL)


## 1. Runtime Check

Run this before downloading data. You should see `CUDA: True` and a `Tesla T4` or similar GPU.


In [ ]:
try:
    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA:", torch.cuda.is_available())
    print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
except Exception as exc:
    print("Torch check failed:", repr(exc))


## 2. Mount Drive

Drive is only needed if you want outputs copied back. Data download happens on Colab local disk by default.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped or failed:", repr(exc))


## 3. Fresh Clone

This resets only the temporary Colab checkout under `/content/human-things`, not your GitHub repo and not Drive.


In [ ]:
def run_checked(command, cwd=None, check=True):
    print("\n$", " ".join(map(str, command)), flush=True)
    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout, flush=True)
    if result.stderr:
        print(result.stderr, flush=True)
    print("exit code:", result.returncode, flush=True)
    if check and result.returncode != 0:
        raise RuntimeError(
            "Command failed with exit code "
            f"{result.returncode}: {' '.join(map(str, command))}"
        )
    return result

print("Step 3: preparing a clean repo checkout", flush=True)
os.chdir("/content")
print("Current working directory:", Path.cwd(), flush=True)

remote_check = run_checked(["git", "ls-remote", "--heads", REPO_URL], check=False)
if remote_check.returncode != 0:
    raise RuntimeError(
        "Colab cannot access the GitHub repo. If the repo is private, make it public "
        "or use an authenticated clone URL. The git error above is the source of truth."
    )

if RESET_LOCAL_REPO:
    print(f"Resetting temporary checkout path: {PROJECT_ROOT}", flush=True)
    shutil.rmtree(PROJECT_ROOT, ignore_errors=True)

if PROJECT_ROOT.exists() and not (PROJECT_ROOT / ".git").exists():
    print(f"{PROJECT_ROOT} exists but is not a git checkout; removing it.", flush=True)
    shutil.rmtree(PROJECT_ROOT, ignore_errors=True)

if not PROJECT_ROOT.exists():
    clone = run_checked(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=False)
    if clone.returncode != 0:
        print("First clone attempt failed. Cleaning target and retrying once.", flush=True)
        shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
        clone = run_checked(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=False)
    if clone.returncode != 0:
        raise RuntimeError("Git clone failed after retry. Read the git output printed above.")
else:
    pull = run_checked(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=False)
    if pull.returncode != 0:
        print("Pull failed. Recreating the temporary checkout from scratch.", flush=True)
        shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
        run_checked(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])

os.chdir(PROJECT_ROOT)
print("Project root:", Path.cwd(), flush=True)
run_checked(["git", "rev-parse", "--short", "HEAD"])
run_checked(["find", "scripts", "-maxdepth", "1", "-type", "f", "-print"])
print("Repo checkout is ready.", flush=True)


## 4. Install Requirements


In [ ]:
print("Step 4: installing Python requirements", flush=True)
print("This can take a few minutes the first time on a new Colab runtime.", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Requirements installed.", flush=True)


## 5. Restore or Download THINGS Data

Preferred path: if `human-things-data.zip` is already in mounted Drive at `MyDrive/human-things-data/human-things-data.zip`, this cell uses it.

Second path: if the mounted file is missing, this cell downloads the shared Drive file by ID to `/content/human-things-data.zip`, then unpacks locally.

Fallback path: if neither Drive zip path works, this cell downloads from OSF. That is slower and can look stuck during large image archive downloads/extraction.

In [ ]:
import zipfile


def restore_zip(zip_path):
    print(f"Restoring data from zip: {zip_path}", flush=True)
    print(f"Zip size GB: {zip_path.stat().st_size / 1e9:.2f}", flush=True)
    data_dir = PROJECT_ROOT / "data"
    if data_dir.exists():
        print(f"Removing existing data folder: {data_dir}", flush=True)
        shutil.rmtree(data_dir)
    print("Unzipping. This can take several minutes for ~28k image files.", flush=True)
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.namelist()
        print(f"Zip entries: {len(members)}", flush=True)
        archive.extractall(PROJECT_ROOT)
    print("Restored:", data_dir, flush=True)
    print("Top-level data folders:", sorted(p.name for p in data_dir.iterdir()), flush=True)


print("Step 5: restoring THINGS data", flush=True)
zip_to_restore = None
if USE_DRIVE_DATA_ZIP and DRIVE_DATA_ZIP.exists():
    print("Found mounted Drive zip.", flush=True)
    zip_to_restore = DRIVE_DATA_ZIP
elif USE_DRIVE_DATA_ZIP and LOCAL_DATA_ZIP.exists():
    print("Found local Colab zip.", flush=True)
    zip_to_restore = LOCAL_DATA_ZIP
elif USE_DRIVE_DATA_ZIP and DRIVE_DATA_FILE_ID:
    print(f"Mounted Drive zip not found: {DRIVE_DATA_ZIP}", flush=True)
    print(f"Downloading shared Drive file to: {LOCAL_DATA_ZIP}", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    subprocess.run([
        sys.executable,
        "-m",
        "gdown",
        "--id",
        DRIVE_DATA_FILE_ID,
        "-O",
        str(LOCAL_DATA_ZIP),
    ], check=True)
    zip_to_restore = LOCAL_DATA_ZIP

if zip_to_restore and zip_to_restore.exists():
    restore_zip(zip_to_restore)
else:
    if USE_DRIVE_DATA_ZIP:
        print("No Drive data zip available. Falling back to OSF download.", flush=True)
    cmd = [sys.executable, "scripts/00_setup_things_data.py"]
    if DOWNLOAD_IMAGES:
        cmd.append("--download-images")
    if FORCE_FETCH:
        cmd.append("--force-fetch")
    print("Running:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
print("Data restore/setup cell finished.", flush=True)


## 6. Verify Processed Data

The important check is that image metadata covers all 1,854 concepts. The earlier broken run had only 1,553 image concepts, which means the raw image metadata was incomplete.


In [ ]:
import pandas as pd

print("Step 6: verifying processed data and image folders", flush=True)
concepts = pd.read_csv(PROJECT_ROOT / "data/processed/concepts.csv")
images = pd.read_csv(PROJECT_ROOT / "data/processed/images.csv")

print("concepts rows:", len(concepts))
print("concept_index unique:", concepts["concept_index"].nunique())
print("images rows:", len(images))
print("image concept_index nulls:", int(images["concept_index"].isna().sum()))
print("image concept_index unique:", images["concept_index"].nunique())

things_dir = PROJECT_ROOT / "data/raw/THINGS-database/osfstorage/images_THINGS/object_images"
cc0_dir = PROJECT_ROOT / "data/raw/THINGS-database/osfstorage/images_THINGSplus-CC0/object_images_CC0"
print("THINGS image dir exists:", things_dir.exists())
print("THINGSplus CC0 image dir exists:", cc0_dir.exists())
print("THINGS jpg count:", sum(1 for _ in things_dir.rglob("*.jpg")) if things_dir.exists() else 0)
print("THINGSplus CC0 jpg count:", sum(1 for _ in cc0_dir.rglob("*.jpg")) if cc0_dir.exists() else 0)

assert len(concepts) == 1854, "Expected 1854 concept rows"
assert concepts["concept_index"].nunique() == 1854, "Expected 1854 unique concept indexes"
assert images["concept_index"].isna().sum() == 0, "Images have missing concept indexes"
assert images["concept_index"].nunique() == 1854, "Image metadata does not cover all 1854 concepts"
assert things_dir.exists(), "Missing extracted THINGS images"


## 7. Build Baseline Metadata and Splits


In [ ]:
print("Step 7: building metadata CSV and within-concept image splits", flush=True)
subprocess.run([sys.executable, "scripts/01_make_metadata_csv.py"], check=True)
subprocess.run([sys.executable, "scripts/02_make_image_splits.py"], check=True)

splits = pd.read_csv(PROJECT_ROOT / "data/baseline/image_splits.csv")
print("split rows:", len(splits), flush=True)
print("concepts:", splits["concept_id"].nunique(), flush=True)
print("missing images:", int((~splits["image_exists"].astype(bool)).sum()), flush=True)
print(splits["split"].value_counts(), flush=True)
assert splits["concept_id"].nunique() == 1854
assert int((~splits["image_exists"].astype(bool)).sum()) == 0
print("Metadata and splits are ready.", flush=True)


## 8. Dry-Run Loader

This loads one batch per split without training. If this works, paths and transforms are OK.


In [ ]:
print("Step 8: dry-running the dataloader", flush=True)
print("This loads one batch from train, val, and test without training.", flush=True)
subprocess.run([
    sys.executable,
    "scripts/03_train_resnet50_image_only.py",
    "--dry-run",
    "--batch-size", "64",
    "--num-workers", "2",
], check=True)
print("Dataloader dry-run passed.", flush=True)


## 9. Short Smoke Training

This is only a GPU sanity check. It intentionally runs a small number of batches, not full epochs over the whole dataset.

In [ ]:
if RUN_SHORT_TRAINING:
    print("Step 9: running bounded smoke training", flush=True)
    print("This runs 20 train batches and 5 eval batches per stage, not full epochs.", flush=True)
    subprocess.run([
        sys.executable,
        "scripts/03_train_resnet50_image_only.py",
        "--head-epochs", "1",
        "--layer4-epochs", "1",
        "--batch-size", "64",
        "--num-workers", "2",
        "--max-train-batches", "20",
        "--max-eval-batches", "5",
    ], check=True)
    print("Smoke training finished.", flush=True)
else:
    print("Skipping short training", flush=True)


## 10. Full Training

Enable only after the short run finishes. Suggested first full run on T4: 5 head epochs and 10 layer4 epochs.


In [ ]:
if RUN_FULL_TRAINING:
    print("Step 10: running full training", flush=True)
    print("This is the real baseline run: 5 head epochs + 10 layer4 epochs.", flush=True)
    subprocess.run([
        sys.executable,
        "scripts/03_train_resnet50_image_only.py",
        "--head-epochs", "5",
        "--layer4-epochs", "10",
        "--batch-size", "64",
        "--num-workers", "2",
    ], check=True)
    print("Full training finished.", flush=True)
else:
    print("Skipping full training", flush=True)


## 11. Extract and Evaluate Embeddings

This uses `outputs/baseline_resnet50/best_model.pt` from whichever training run completed most recently.


In [ ]:
if RUN_EXTRACT_AND_EVAL:
    print("Step 11: extracting embeddings and running baseline evaluations", flush=True)
    subprocess.run([
        sys.executable,
        "scripts/04_extract_resnet50_embeddings.py",
        "--batch-size", "128",
        "--num-workers", "2",
    ], check=True)
    subprocess.run([sys.executable, "scripts/05_evaluate_resnet50_embeddings.py"], check=True)
    print("Embedding extraction/evaluation finished.", flush=True)
else:
    print("Skipping extraction/evaluation", flush=True)


## 12. Copy Outputs to Drive

This keeps checkpoints, logs, metadata, and evaluation files after the Colab runtime disappears.


In [ ]:
if COPY_OUTPUTS_TO_DRIVE and Path("/content/drive/MyDrive").exists():
    print("Step 12: copying outputs back to Drive", flush=True)
    run_dir = DRIVE_OUTPUT_ROOT / "baseline_resnet50"
    run_dir.mkdir(parents=True, exist_ok=True)
    for name in ["outputs", "data/baseline", "data/processed"]:
        src = PROJECT_ROOT / name
        dst = run_dir / name
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"Copied {src} -> {dst}", flush=True)
    print("Drive copy finished:", run_dir, flush=True)
else:
    print("Skipping Drive copy", flush=True)
